In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from yellowbrick.cluster import SilhouetteVisualizer

### Leitura de dados

In [ ]:
df = pd.read_csv('../dados_processados/cursos-pos-processamento.csv',sep=';')

In [ ]:
df_scaled = df.drop(['no_curso','co_curso'],axis=1)

### Funções

In [ ]:
random_state = 42

In [ ]:
def plotar_grafico(sum_of_squares, numero_componentes):
    plt.figure(figsize = (10, 8))
    plt.plot(range(2, 21), sum_of_squares, marker = 'o', linestyle = '--')
    plt.xlabel("Número de clusters")
    plt.ylabel("Soma dos Quadrados Dentro do Cluster (WCSS)")
    plt.title("K-means após utilização de PCA - nº de componentes: " + str(numero_componentes))


In [ ]:
def visualizar_silhueta_k_means(kmeans, data):
    visualizer = SilhouetteVisualizer(kmeans)
    visualizer.fit(data)
    silhouette_score_value = visualizer.silhouette_score_
    print(f"Silhouette Visualizer (Score: {silhouette_score_value:.2f})")
    visualizer.show()

In [ ]:
def visualizar_clusters(kmeans, data):
    # Labels dos clusters
    labels = kmeans.labels_
    # Centros dos clusters
    centroids = kmeans.cluster_centers_
    # Plotando os dados e os clusters
    plt.figure(figsize=(10, 6))
    # Plotando os pontos com cores baseadas nos clusters
    plt.scatter(data[:, 0], data[:, 1], c=labels, cmap='viridis', marker='o', s=100, edgecolor='k')
    # Plotando os centróides
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=200, label='Centroides')
    
    plt.title('Clusters com PCA + K-Means')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def plotar_resultados_k_means(n_clusters, scores_pca, n_components):
    print("Número de componentes utilizado " + str(n_components))
    print()
    data_pca = reduzir_dimensionalidade_PCA(n_components)
    k_means = aplicar_k_means(data_pca, n_clusters)
    visualizar_silhueta_k_means(k_means, data_pca)
    visualizar_clusters(k_means, data_pca)

In [ ]:
def aplicar_k_means(data, n_clusters):
    #Aplicar K-Means no espaço reduzido
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(data)

    return kmeans

In [ ]:
def reduzir_dimensionalidade_PCA(n_components):
    #Reduzir dimensionalidade com PCA
    pca = PCA(n_components=n_components, random_state=random_state)
    data_pca = pca.fit_transform(df_scaled)

    return data_pca

In [ ]:
def analisar_wcss(componente_inicial, componente_final, k_inicial, k_final):
    for n_components in range(componente_inicial, componente_final+1):
        data_pca = reduzir_dimensionalidade_PCA(n_components)
        
        wcss = []
        for k in range(k_inicial, k_final+1):
            kmeans_pca = aplicar_k_means(data_pca, k)
            wcss.append(kmeans_pca.inertia_)
    
        plotar_grafico(wcss, n_components)

In [ ]:
def analisar_silhueta(componente_inicial, componente_final, k_inicial, k_final):
     for n_components in range(componente_inicial, componente_final+1):
         data_pca = reduzir_dimensionalidade_PCA(n_components)
         
         for k in range(k_inicial, k_final+1):
            kmeans_pca = aplicar_k_means(data_pca, k)
            plotar_resultados_k_means(k, data_pca, n_components)

### Análise - Variância explicada por componentes

In [ ]:
pca_geral = PCA()
pca_geral.fit(df_scaled)

In [ ]:
pca_geral.explained_variance_ratio_

In [ ]:
plt.figure(figsize = (10, 8))
plt.plot(range(1, 96), pca_geral.explained_variance_ratio_.cumsum(), marker = 'o', linestyle = '--')
plt.title("Variância explicada por componentes")
plt.xlabel("Número de componentes")
plt.ylabel("Variância cumulativa explicada")

### Análise WCSS

In [ ]:
analisar_wcss(2, 40, 2, 20)

### Análise Silhueta

In [ ]:
analisar_silhueta(2, 40, 2, 10)

In [ ]:
print("Análise finalizada")